# EEG-only experiments (text2text template)

This notebook is adapted from the text2text workflow but uses only EEG features.
Text2text features are not concatenated into the training matrix.

Targets/splits covered:
- `match_mismatch`: binary + multiclass
- `match_mismatch_general`: binary + multiclass

In [1]:
from __future__ import annotations

import os
import json
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from umap import UMAP

from experiment_code.read_data import get_data_for_split
from experiment_code.run_experiments import _build_groups_mm_for_mm
from experiment_code.optuna_code import run_and_log, fit_best_and_test

# Data root: contains datasets_splitted_screen_match_mismatch*
ROOT = Path(r"C:\Users\LEGION\data\СB")
os.chdir(ROOT)
print("cwd:", Path.cwd())

cwd: C:\Users\LEGION\data\СB


In [2]:
# Optuna / CV settings aligned with your ES experiments
TARGETS = ["match_mismatch", "match_mismatch_general"]
SPLITS = ["binary", "multiclass"]

USE_EARLY_STOPPING = True
REFIT_CV = False
REFIT_TEST = True
USE_MIN = False

CV = 4
N_TRIALS_XGB = 100
N_TRIALS_CB = 50

# Output location under project folder (not ROOT)
PROJECT_DIR = Path(r"C:\Users\LEGION\Projects\CB_exepriment\dataset_v2")
OUT_DIR = PROJECT_DIR / "optuna_results_eye_text2text"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FI_DIR = OUT_DIR / "feature_importances_eye_text2text"
FI_DIR.mkdir(parents=True, exist_ok=True)
REFIT_TOP_N = 30

# Compress only these text2text sets with UMAP (fit on train, transform on test)
UMAP_TARGET_SETS = {"freq_bands", "stat", "corr", "cov_freq"}
UMAP_N_COMPONENTS = 100
# Keep only columns with no NaN in train (same style as previous experiments)
CLEAN_TRAIN_COLUMNS = False

In [ ]:
def _prefix_cols(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return df.add_prefix(prefix)


def _load_text2text_parquet(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    if "pid_rn" not in df.columns:
        raise KeyError(f"'pid_rn' column not found in {path}")
    df = df.set_index("pid_rn")
    df.index = df.index.astype(str)
    return df


def _apply_umap_train_test(
    text_train: pd.DataFrame,
    text_test: pd.DataFrame,
    *,
    text_set_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    tr = text_train.apply(pd.to_numeric, errors="coerce")
    te = text_test.apply(pd.to_numeric, errors="coerce")

    train_means = tr.mean(numeric_only=True)
    tr = tr.fillna(train_means).fillna(0.0)
    te = te.fillna(train_means).fillna(0.0)

    n_components = int(min(UMAP_N_COMPONENTS, max(2, tr.shape[1])))
    reducer = UMAP(
        n_components=n_components,
    )

    print(
        f"[{text_set_name}] UMAP compression: "
        f"train/test features {tr.shape[1]} -> {n_components}"
    )

    z_train = reducer.fit_transform(tr)
    z_test = reducer.transform(te)

    cols = [f"{text_set_name}__umap_{i:03d}" for i in range(n_components)]
    tr_umap = pd.DataFrame(z_train, index=text_train.index, columns=cols)
    te_umap = pd.DataFrame(z_test, index=text_test.index, columns=cols)
    return tr_umap, te_umap


def _prepare_joined_X(
    train_index: pd.Index,
    test_index: pd.Index,
    text_all: pd.DataFrame,
    text_set_name: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Index match because text2text is not pre-split
    text_train = text_all.reindex(train_index)
    text_test = text_all.reindex(test_index)

    missing_train = int(text_train.isna().all(axis=1).sum())
    missing_test = int(text_test.isna().all(axis=1).sum())
    print(f"[{text_set_name}] rows missing after index match -> train: {missing_train}, test: {missing_test}")

    if text_set_name in UMAP_TARGET_SETS:
        text_train_model, text_test_model = _apply_umap_train_test(
            text_train,
            text_test,
            text_set_name=text_set_name,
        )
    else:
        text_train_model = _prefix_cols(text_train, f"{text_set_name}__")
        text_test_model = _prefix_cols(text_test, f"{text_set_name}__")

    # EEG-only setup: use only EEG-set features from the selected parquet.
    X_train = text_train_model
    X_test = text_test_model

    if CLEAN_TRAIN_COLUMNS:
        keep_cols = X_train.columns[X_train.notna().all(axis=0)]
        X_train = X_train[keep_cols]
        X_test = X_test.reindex(columns=keep_cols)

    return X_train, X_test


def _build_no_neutral_mask(stim_df: pd.DataFrame, target: str, split: str) -> pd.Series:
    mask = pd.Series(True, index=stim_df.index)

    if "valence" in stim_df.columns:
        valence = pd.to_numeric(stim_df["valence"], errors="coerce")
        mask &= valence != 2

    if split == "binary":
        if target == "match_mismatch_general" and "exp_general_multi" in stim_df.columns:
            exp_general_multi = pd.to_numeric(stim_df["exp_general_multi"], errors="coerce")
            mask &= exp_general_multi != 2
        elif "IAT_results2" in stim_df.columns:
            iat = stim_df["IAT_results2"].astype(str).str.strip().str.lower()
            mask &= iat != "neutral"

    return mask.fillna(False)


def _load_eeg_for_task(target: str, split: str, no_neutral: bool = False):
    df, tgt, _ = get_data_for_split(X_name="screen", target=target, split=split)

    train_index = df["all_features"]["X_train"].index.astype(str)
    test_index = df["all_features"]["X_test"].index.astype(str)
    stim_train = df["stimuli_features"]["X_train"].copy()
    stim_test = df["stimuli_features"]["X_test"].copy()

    y_train = tgt["cb"]["y_train"].copy()
    y_test = tgt["cb"]["y_test"].copy()

    if no_neutral:
        mask_train = _build_no_neutral_mask(stim_train, target=target, split=split)
        mask_test = _build_no_neutral_mask(stim_test, target=target, split=split)

        train_index = train_index[mask_train.to_numpy()]
        test_index = test_index[mask_test.to_numpy()]
        y_train = y_train.loc[mask_train]
        y_test = y_test.loc[mask_test]
        stim_train = stim_train.loc[mask_train]

    groups = np.array([str(i).split("_")[0] for i in train_index])

    if target in ("match_mismatch", "match_mismatch_general"):
        groups_mm = _build_groups_mm_for_mm(X_train_index=train_index, stim_train_df=stim_train)
    else:
        groups_mm = None

    problem = "binary" if split == "binary" else "multiclass"
    return train_index, test_index, y_train, y_test, groups, groups_mm, problem


def run_text2text_set(
    text_set_name: str,
    text_all: pd.DataFrame,
    *,
    do_refit: bool = True,
    top_n_features: int = REFIT_TOP_N,
):
    # Enforce index used for matching with eye-tracking splits.
    if "pid_rn" in text_all.columns:
        text_all = text_all.set_index("pid_rn")
    text_all = text_all.copy()
    text_all.index = text_all.index.astype(str)

    # 'key' must never be used as a feature.
    if "key" in text_all.columns:
        text_all = text_all.drop(columns=["key"])

    text_features = int(text_all.shape[1])
    print(f"[{text_set_name}] text2text feature count after cleanup: {text_features}")

    rows = []

    for target in TARGETS:
        for split in SPLITS:
            for no_neutral in [False, True]:
                problem = "binary" if split == "binary" else "multiclass"
                neutral_suffix = "__no_neutral" if no_neutral else ""
                train_features_name = (
                    f"exp__X_name=screen+{text_set_name}"
                    f"__target={target}"
                    f"__problem={problem}"
                    f"__feat=all_features+{text_set_name}"
                    f"__ES__refitTEST"
                    f"{neutral_suffix}"
                )
                out_path = OUT_DIR / f"{train_features_name}.json"

                if out_path.exists():
                    print("\n" + "=" * 80)
                    print(f"Skipping existing experiment: {train_features_name}")
                    with open(out_path, encoding="utf-8") as f:
                        results = json.load(f)
                    rows.append({
                        "text_set": text_set_name,
                        "target": target,
                        "split": split,
                        "no_neutral": no_neutral,
                        "xgb_test": float(results["xgb"]["test_metrics"]["primary"]),
                        "catboost_test": float(results["catboost"]["test_metrics"]["primary"]),
                        "file": str(out_path),
                    })
                    continue

                train_index, test_index, y_train, y_test, groups, groups_mm, problem = _load_eeg_for_task(
                    target,
                    split,
                    no_neutral=no_neutral,
                )
                X_train, X_test = _prepare_joined_X(train_index, test_index, text_all, text_set_name)

                print("\n" + "=" * 80)
                print(f"Running: {train_features_name}")
                print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

                results = run_and_log(
                    train_features=train_features_name,
                    problem=problem,
                    X_train=X_train,
                    X_test=X_test,
                    y_train=y_train,
                    y_test=y_test,
                    strat_train=y_train,
                    groups=groups,
                    groups_mm=groups_mm,
                    use_early_stopping=USE_EARLY_STOPPING,
                    refit_cv=REFIT_CV,
                    refit_test=REFIT_TEST,
                    n_trials_xgb=N_TRIALS_XGB,
                    n_trials_cb=N_TRIALS_CB,
                    cv=CV,
                    gpu=False,
                    use_min=USE_MIN,
                    cat_cols=None,
                    target_names=None,
                    out_path=out_path,
                )

                if do_refit:
                    print("\n" + "-" * 80)
                    print("Refit best iteration + metrics + top features")
                    for model_name in ["xgb", "catboost"]:
                        best_params = results[model_name].get("suggested_params", {})
                        if not best_params:
                            print(f"\n=== Skip refit {model_name.upper()} (no suggested params) ===")
                            continue

                        print(f"\n=== Refit {model_name.upper()} with saved best params ===")
                        out_refit = fit_best_and_test(
                            model_name=model_name,
                            best_params=best_params,
                            problem=problem,
                            X_train=X_train,
                            X_test=X_test,
                            y_train=y_train,
                            y_test=y_test,
                            strat_train=y_train,
                            groups=groups,
                            groups_mm=groups_mm,
                            use_early_stopping=True,
                            early_stopping_rounds=100,
                            target_names=None,
                            cat_cols=None,
                            gpu=False,
                            refit_test=True,
                        )

                        model = out_refit["model"]
                        if model_name == "xgb":
                            importances = model.feature_importances_
                        else:
                            importances = model.get_feature_importance()

                        fi = pd.DataFrame({
                            "feature": X_train.columns,
                            "importance": importances,
                        }).sort_values("importance", ascending=False)

                        fi_path = FI_DIR / f"{train_features_name}__{model_name}.csv"
                        fi.to_csv(fi_path, index=False)
                        print(f"Saved feature importances: {fi_path}")
                        print(f"Top {top_n_features} features for {model_name.upper()}:")
                        display(fi.head(top_n_features))

                rows.append({
                    "text_set": text_set_name,
                    "target": target,
                    "split": split,
                    "no_neutral": no_neutral,
                    "xgb_test": float(results["xgb"]["test_metrics"]["primary"]),
                    "catboost_test": float(results["catboost"]["test_metrics"]["primary"]),
                    "file": str(out_path),
                })

    summary = pd.DataFrame(rows)
    display(summary)
    return summary

In [4]:
# Explicit text2text parquet sets
TEXT2TEXT_DIR = Path(r"C:\Users\LEGION\Projects\CB_exepriment\text2text_features")
TEXT2TEXT_FILES = {
    "corr": TEXT2TEXT_DIR / "corr.parquet",
    "cov_freq": TEXT2TEXT_DIR / "cov_freq.parquet",
    "envelope": TEXT2TEXT_DIR / "envelope.parquet",
    "freq_bands": TEXT2TEXT_DIR / "freq_bands.parquet",
    "PID": TEXT2TEXT_DIR / "PID.parquet",
    "stat": TEXT2TEXT_DIR / "stat.parquet",
}

print("text2text dir:", TEXT2TEXT_DIR)
for k, p in TEXT2TEXT_FILES.items():
    print(f"{k:10s} -> {p} | exists={p.exists()}")
    print(pd.read_parquet(p).shape)

text2text dir: C:\Users\LEGION\Projects\CB_exepriment\text2text_features
corr       -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\corr.parquet | exists=True
(5592, 1832)
cov_freq   -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\cov_freq.parquet | exists=True
(5592, 36297)
envelope   -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\envelope.parquet | exists=True
(5592, 429)
freq_bands -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\freq_bands.parquet | exists=True
(5592, 18363)
PID        -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\PID.parquet | exists=True
(5592, 612)
stat       -> C:\Users\LEGION\Projects\CB_exepriment\text2text_features\stat.parquet | exists=True
(5592, 8237)


In [5]:
# Set: corr
text_corr = pd.read_parquet(TEXT2TEXT_FILES["corr"])
print("corr shape:", text_corr.shape)
summary_corr = run_text2text_set("corr", text_corr)
summary_corr

corr shape: (5592, 1832)
[corr] text2text feature count after cleanup: 1830

Skipping existing experiment: exp__X_name=screen+corr__target=match_mismatch__problem=binary__feat=all_features+corr__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+corr__target=match_mismatch__problem=multiclass__feat=all_features+corr__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+corr__target=match_mismatch_general__problem=binary__feat=all_features+corr__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+corr__target=match_mismatch_general__problem=multiclass__feat=all_features+corr__ES__refitTEST


,text_set,target,split,xgb_test,catboost_test,file
0,corr,match_mismatch,binary,0.399740,0.400511,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,multiclass,0.621334,0.669227,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch_general,binary,0.406654,0.532466,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch_general,multiclass,0.797868,0.794106,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,xgb_test,catboost_test,file
0,corr,match_mismatch,binary,0.399740,0.400511,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,multiclass,0.621334,0.669227,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch_general,binary,0.406654,0.532466,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch_general,multiclass,0.797868,0.794106,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [6]:
# Set: cov_freq
text_cov_freq = pd.read_parquet(TEXT2TEXT_FILES["cov_freq"])
print("cov_freq shape:", text_cov_freq.shape)
summary_cov_freq = run_text2text_set("cov_freq", text_cov_freq)
summary_cov_freq

cov_freq shape: (5592, 36297)
[cov_freq] text2text feature count after cleanup: 36295

Skipping existing experiment: exp__X_name=screen+cov_freq__target=match_mismatch__problem=binary__feat=all_features+cov_freq__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+cov_freq__target=match_mismatch__problem=multiclass__feat=all_features+cov_freq__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+cov_freq__target=match_mismatch_general__problem=binary__feat=all_features+cov_freq__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+cov_freq__target=match_mismatch_general__problem=multiclass__feat=all_features+cov_freq__ES__refitTEST


,text_set,target,split,xgb_test,catboost_test,file
0,cov_freq,match_mismatch,binary,0.399740,0.415582,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,cov_freq,match_mismatch,multiclass,0.619349,0.626280,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,cov_freq,match_mismatch_general,binary,0.475352,0.579789,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,cov_freq,match_mismatch_general,multiclass,0.801701,0.803497,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,xgb_test,catboost_test,file
0,cov_freq,match_mismatch,binary,0.399740,0.415582,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,cov_freq,match_mismatch,multiclass,0.619349,0.626280,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,cov_freq,match_mismatch_general,binary,0.475352,0.579789,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,cov_freq,match_mismatch_general,multiclass,0.801701,0.803497,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [7]:
# Set: envelope
text_envelope = pd.read_parquet(TEXT2TEXT_FILES["envelope"])
print("envelope shape:", text_envelope.shape)
summary_envelope = run_text2text_set("envelope", text_envelope)
summary_envelope

envelope shape: (5592, 429)
[envelope] text2text feature count after cleanup: 427

Skipping existing experiment: exp__X_name=screen+envelope__target=match_mismatch__problem=binary__feat=all_features+envelope__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+envelope__target=match_mismatch__problem=multiclass__feat=all_features+envelope__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+envelope__target=match_mismatch_general__problem=binary__feat=all_features+envelope__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+envelope__target=match_mismatch_general__problem=multiclass__feat=all_features+envelope__ES__refitTEST


,text_set,target,split,xgb_test,catboost_test,file
0,envelope,match_mismatch,binary,0.399740,0.436447,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,envelope,match_mismatch,multiclass,0.632589,0.659056,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,envelope,match_mismatch_general,binary,0.398957,0.504534,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,envelope,match_mismatch_general,multiclass,0.812691,0.803043,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,xgb_test,catboost_test,file
0,envelope,match_mismatch,binary,0.399740,0.436447,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,envelope,match_mismatch,multiclass,0.632589,0.659056,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,envelope,match_mismatch_general,binary,0.398957,0.504534,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,envelope,match_mismatch_general,multiclass,0.812691,0.803043,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [8]:
# Set: freq_bands
text_freq_bands = pd.read_parquet(TEXT2TEXT_FILES["freq_bands"])
print("freq_bands shape:", text_freq_bands.shape)
summary_freq_bands = run_text2text_set("freq_bands", text_freq_bands)
summary_freq_bands

freq_bands shape: (5592, 18363)
[freq_bands] text2text feature count after cleanup: 18361

Skipping existing experiment: exp__X_name=screen+freq_bands__target=match_mismatch__problem=binary__feat=all_features+freq_bands__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+freq_bands__target=match_mismatch__problem=multiclass__feat=all_features+freq_bands__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+freq_bands__target=match_mismatch_general__problem=binary__feat=all_features+freq_bands__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+freq_bands__target=match_mismatch_general__problem=multiclass__feat=all_features+freq_bands__ES__refitTEST


,text_set,target,split,xgb_test,catboost_test,file
0,freq_bands,match_mismatch,binary,0.399740,0.413223,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,freq_bands,match_mismatch,multiclass,0.639669,0.620434,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,freq_bands,match_mismatch_general,binary,0.398957,0.509769,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,freq_bands,match_mismatch_general,multiclass,0.806212,0.792472,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,xgb_test,catboost_test,file
0,freq_bands,match_mismatch,binary,0.399740,0.413223,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,freq_bands,match_mismatch,multiclass,0.639669,0.620434,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,freq_bands,match_mismatch_general,binary,0.398957,0.509769,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,freq_bands,match_mismatch_general,multiclass,0.806212,0.792472,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [9]:
# Set: PID
text_PID = pd.read_parquet(TEXT2TEXT_FILES["PID"])
print("PID shape:", text_PID.shape)
summary_PID = run_text2text_set("PID", text_PID)
summary_PID

PID shape: (5592, 612)
[PID] text2text feature count after cleanup: 610

Skipping existing experiment: exp__X_name=screen+PID__target=match_mismatch__problem=binary__feat=all_features+PID__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+PID__target=match_mismatch__problem=multiclass__feat=all_features+PID__ES__refitTEST

Skipping existing experiment: exp__X_name=screen+PID__target=match_mismatch_general__problem=binary__feat=all_features+PID__ES__refitTEST


[I 2026-04-09 11:25:19,590] A new study created in memory with name: multiclass_xgb


[PID] rows missing after index match -> train: 1193, test: 131

Running: exp__X_name=screen+PID__target=match_mismatch_general__problem=multiclass__feat=all_features+PID__ES__refitTEST
X_train: (4146, 1080), X_test: (461, 1080)


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-09 11:25:53,170] Trial 0 finished with value: 0.6570647149336668 and parameters: {'learning_rate': 0.04534239315199742, 'max_depth': 4, 'min_child_weight': 3.6133378418934727, 'subsample': 0.7172121707956832, 'colsample_bytree': 0.7642269046455532, 'gamma': 0.2930268354654828, 'reg_alpha': 0.0005749550436838274, 'reg_lambda': 2.9181778067053106}. Best is trial 0 with value: 0.6570647149336668.
[I 2026-04-09 11:26:16,535] Trial 1 finished with value: 0.6509148244905224 and parameters: {'learning_rate': 0.1707594054749505, 'max_depth': 8, 'min_child_weight': 1.8407677988385938, 'subsample': 0.9514458866935752, 'colsample_bytree': 0.7320289895353271, 'gamma': 0.030927525032881986, 'reg_alpha': 2.3384483013779853, 'reg_lambda': 0.0007548524119611296}. Best is trial 0 with value: 0.6570647149336668.
[I 2026-04-09 11:26:51,327] Trial 2 finished with value: 0.657240097940272 and parameters: {'learning_rate': 0.0333616681035175, 'max_depth': 5, 'min_child_weight': 1.3687627858565237

...

,feature,importance
276,eye__fix_duration_AOI_aggregated_Pos_min,0.022874
184,eye__fix_duration_AOI_FD_Neg1_count,0.022613
29,eye__sac_length_AOI_aggregated_Neg_min,0.020072
177,eye__fix_duration_AOI_FD_Neg1_max,0.016650
62,eye__sac_speed_AOI_aggregated_Pos_mean,0.016568
30,eye__sac_length_AOI_aggregated_Neg_max,0.014650
61,eye__sac_speed_AOI_aggregated_Pos_max,0.013592
175,eye__fix_duration_AOI_FD_Neg1_first,0.012613
54,eye__sac_length_AOI_aggregated_Pos_max,0.012601
63,eye__sac_speed_AOI_aggregated_Pos_median,0.012439



=== Refit CATBOOST with saved best params ===

[catboost] Test macro-F1: 0.77404
              precision    recall  f1-score   support

           0       0.77      0.63      0.70       156
           1       0.80      0.74      0.77       180
           2       0.75      1.00      0.86       125

    accuracy                           0.77       461
   macro avg       0.77      0.79      0.77       461
weighted avg       0.78      0.77      0.77       461

Confusion matrix:
 [[ 99  33  24]
 [ 29 133  18]
 [  0   0 125]]
Saved feature importances: C:\Users\LEGION\Projects\CB_exepriment\dataset_v2\optuna_results_eye_text2text\feature_importances_eye_text2text\exp__X_name=screen+PID__target=match_mismatch_general__problem=multiclass__feat=all_features+PID__ES__refitTEST__catboost.csv
Top 30 features for CATBOOST:


,feature,importance
258,eye__fix_duration_AOI_aggregated_Neg_mean,9.757130
39,eye__sac_speed_AOI_aggregated_Neg_median,6.285012
35,eye__sac_length_AOI_aggregated_Neg_count,5.699372
30,eye__sac_length_AOI_aggregated_Neg_max,4.912150
175,eye__fix_duration_AOI_FD_Neg1_first,4.503257
32,eye__sac_length_AOI_aggregated_Neg_median,4.222576
176,eye__fix_duration_AOI_FD_Neg1_min,3.514817
256,eye__fix_duration_AOI_aggregated_Neg_min,3.190887
34,eye__sac_length_AOI_aggregated_Neg_sum,3.178013
184,eye__fix_duration_AOI_FD_Neg1_count,2.762312


,text_set,target,split,xgb_test,catboost_test,file
0,PID,match_mismatch,binary,0.399740,0.443317,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,PID,match_mismatch,multiclass,0.627365,0.624341,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,PID,match_mismatch_general,binary,0.398957,0.478687,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,PID,match_mismatch_general,multiclass,0.795993,0.774045,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,xgb_test,catboost_test,file
0,PID,match_mismatch,binary,0.399740,0.443317,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,PID,match_mismatch,multiclass,0.627365,0.624341,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,PID,match_mismatch_general,binary,0.398957,0.478687,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,PID,match_mismatch_general,multiclass,0.795993,0.774045,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [10]:
# Set: stat
text_stat = pd.read_parquet(TEXT2TEXT_FILES["stat"])
print("stat shape:", text_stat.shape)
summary_stat = run_text2text_set("stat", text_stat)
summary_stat

stat shape: (5592, 8237)
[stat] text2text feature count after cleanup: 8235
[stat] rows missing after index match -> train: 1193, test: 131
[stat] UMAP compression: train/test features 8235 -> 100


[I 2026-04-09 13:52:39,992] A new study created in memory with name: binary_xgb



Running: exp__X_name=screen+stat__target=match_mismatch__problem=binary__feat=all_features+stat__ES__refitTEST
X_train: (4146, 570), X_test: (461, 570)


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-09 13:52:44,604] Trial 0 finished with value: 0.4007818691904268 and parameters: {'learning_rate': 0.04534239315199742, 'max_depth': 4, 'min_child_weight': 3.6133378418934727, 'subsample': 0.7172121707956832, 'colsample_bytree': 0.7642269046455532, 'gamma': 0.2930268354654828, 'reg_alpha': 0.0005749550436838274, 'reg_lambda': 2.9181778067053106}. Best is trial 0 with value: 0.4007818691904268.
[I 2026-04-09 13:52:50,560] Trial 1 finished with value: 0.4007818691904268 and parameters: {'learning_rate': 0.1707594054749505, 'max_depth': 8, 'min_child_weight': 1.8407677988385938, 'subsample': 0.9514458866935752, 'colsample_bytree': 0.7320289895353271, 'gamma': 0.030927525032881986, 'reg_alpha': 2.3384483013779853, 'reg_lambda': 0.0007548524119611296}. Best is trial 0 with value: 0.4007818691904268.
[I 2026-04-09 13:52:56,167] Trial 2 finished with value: 0.4007818691904268 and parameters: {'learning_rate': 0.0333616681035175, 'max_depth': 5, 'min_child_weight': 1.368762785856523

...

,feature,importance
276,eye__fix_duration_AOI_aggregated_Pos_min,0.049305
30,eye__sac_length_AOI_aggregated_Neg_max,0.029191
62,eye__sac_speed_AOI_aggregated_Pos_mean,0.027826
55,eye__sac_length_AOI_aggregated_Pos_mean,0.026835
176,eye__fix_duration_AOI_FD_Neg1_min,0.024355
29,eye__sac_length_AOI_aggregated_Neg_min,0.024344
177,eye__fix_duration_AOI_FD_Neg1_max,0.023283
56,eye__sac_length_AOI_aggregated_Pos_median,0.018503
184,eye__fix_duration_AOI_FD_Neg1_count,0.018419
175,eye__fix_duration_AOI_FD_Neg1_first,0.015231



=== Refit CATBOOST with saved best params ===

[catboost] Test macro-F1: 0.80777
              precision    recall  f1-score   support

           0       0.85      0.67      0.75       156
           1       0.84      0.79      0.82       180
           2       0.75      1.00      0.86       125

    accuracy                           0.81       461
   macro avg       0.81      0.82      0.81       461
weighted avg       0.82      0.81      0.81       461

Confusion matrix:
 [[105  27  24]
 [ 19 143  18]
 [  0   0 125]]
Saved feature importances: C:\Users\LEGION\Projects\CB_exepriment\dataset_v2\optuna_results_eye_text2text\feature_importances_eye_text2text\exp__X_name=screen+stat__target=match_mismatch_general__problem=multiclass__feat=all_features+stat__ES__refitTEST__catboost.csv
Top 30 features for CATBOOST:


,feature,importance
256,eye__fix_duration_AOI_aggregated_Neg_min,6.327715
35,eye__sac_length_AOI_aggregated_Neg_count,5.948342
259,eye__fix_duration_AOI_aggregated_Neg_median,5.646431
264,eye__fix_duration_AOI_aggregated_Neg_count,5.397057
32,eye__sac_length_AOI_aggregated_Neg_median,4.918009
38,eye__sac_speed_AOI_aggregated_Neg_mean,4.581234
36,eye__sac_speed_AOI_aggregated_Neg_min,4.244335
255,eye__fix_duration_AOI_aggregated_Neg_first,3.169397
263,eye__fix_duration_AOI_aggregated_Neg_sum,3.142982
31,eye__sac_length_AOI_aggregated_Neg_mean,2.860741


,text_set,target,split,xgb_test,catboost_test,file
0,stat,match_mismatch,binary,0.399740,0.425346,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,stat,match_mismatch,multiclass,0.638100,0.647005,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,stat,match_mismatch_general,binary,0.449997,0.585391,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,stat,match_mismatch_general,multiclass,0.816845,0.807769,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,text_set,target,split,xgb_test,catboost_test,file
0,stat,match_mismatch,binary,0.399740,0.425346,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,stat,match_mismatch,multiclass,0.638100,0.647005,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,stat,match_mismatch_general,binary,0.449997,0.585391,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,stat,match_mismatch_general,multiclass,0.816845,0.807769,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [11]:
# Combine summaries from all explicit sets
all_summaries = [
    summary_corr,
    summary_cov_freq,
    summary_envelope,
    summary_freq_bands,
    summary_PID,
    summary_stat,
]
all_results = pd.concat(all_summaries, ignore_index=True)
all_results

,text_set,target,split,xgb_test,catboost_test,file
0,corr,match_mismatch,binary,0.399740,0.400511,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,multiclass,0.621334,0.669227,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch_general,binary,0.406654,0.532466,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch_general,multiclass,0.797868,0.794106,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,cov_freq,match_mismatch,binary,0.399740,0.415582,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,cov_freq,match_mismatch,multiclass,0.619349,0.626280,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,cov_freq,match_mismatch_general,binary,0.475352,0.579789,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,cov_freq,match_mismatch_general,multiclass,0.801701,0.803497,C:\Users\LEGION\Projects\CB_exepriment\dataset...
8,envelope,match_mismatch,binary,0.399740,0.436447,C:\Users\LEGION\Projects\CB_exepriment\dataset...
9,envelope,match_mismatch,multiclass,0.632589,0.659056,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [12]:
# Save + quick inspection
summary_path = OUT_DIR / "summary_eye_text2text.csv"
all_results.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)

all_results.groupby(["text_set", "target", "split"], as_index=False)[["xgb_test", "catboost_test"]].mean()

Saved summary: C:\Users\LEGION\Projects\CB_exepriment\dataset_v2\optuna_results_eye_text2text\summary_eye_text2text.csv


,text_set,target,split,xgb_test,catboost_test
0,PID,match_mismatch,binary,0.399740,0.443317
1,PID,match_mismatch,multiclass,0.627365,0.624341
2,PID,match_mismatch_general,binary,0.398957,0.478687
3,PID,match_mismatch_general,multiclass,0.795993,0.774045
4,corr,match_mismatch,binary,0.399740,0.400511
5,corr,match_mismatch,multiclass,0.621334,0.669227
6,corr,match_mismatch_general,binary,0.406654,0.532466
7,corr,match_mismatch_general,multiclass,0.797868,0.794106
8,cov_freq,match_mismatch,binary,0.399740,0.415582
9,cov_freq,match_mismatch,multiclass,0.619349,0.626280
